# Phase 12: Tracing and evals

**Tracing** records every model call and step of a run, so you can see what happened. **Evals** score the agents on labelled reports, so you know whether a change made things better or worse.

Before you run the cells: `.env` must contain `LANGSMITH_API_KEY`, `LANGSMITH_TRACING=true` and `LANGSMITH_PROJECT=community-voice`. Tracing sends report text to LangSmith, so use fictional data only.

In [ ]:
import json
import sys

import langsmith as ls

from community_voice.core.config import PROJECT_ROOT
from community_voice.core.tracing import load_env

sys.path.insert(0, str(PROJECT_ROOT))

from evals import scoring  # noqa: E402
from evals.run_evals import BASELINE, load_examples, target  # noqa: E402

load_env()  # puts .env into the environment so LangSmith can see it

## 1. One traced run

Triage and routing run on one labelled report (live wires near a hospital). Then open the `community-voice` project at smith.langchain.com and click the newest trace: you will see each model call and tool call.

In [ ]:
example = load_examples()[4]
with ls.tracing_context(enabled=True, metadata={"demo": "phase-12"}):
    output = target(example["inputs"])
output

## 2. Score it

Each score compares the agent's answer with the label. A score of `not scored` means it does not apply to this report.

In [ ]:
for evaluator in scoring.EVALUATORS:
    result = evaluator(output, example["outputs"])
    print(f"{evaluator.__name__:20}", result.get("score", "not scored"))

## 3. The regression check

The baseline holds the scores we accepted. A run fails if a score drops more than 0.10 below it. Here we pretend a prompt change broke categories.

In [ ]:
baseline = json.loads(BASELINE.read_text())
worse = {**baseline, "category": 0.6}
print("Pretend run:", scoring.regressions(worse, baseline))
print("Same as baseline:", scoring.regressions(baseline, baseline))

## 4. The full evaluation

Run all 20 reports from a terminal (this takes a few minutes):

```bash
uv run python -m evals.run_evals --refresh
```

Open the link it prints to see each report's result in LangSmith. Run it again after a change and compare the two experiments side by side.

## What to notice

- A trace shows how an answer was produced, not just the answer.
- The "missed" lines show exactly where the agent and the labels disagree.
- A low score can mean a bad label or a bad scorer, not only a bad agent. Check before blaming the agent.